[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/service_assistant/openai_agents.ipynb)

# Simulated service assistant — OpenAI Agents SDK

**Task.** Safely update one simulated order address using typed request-review and confirmation agents.

**Read this notebook as:** choose a provider → load the simulation → declare the SDK adapter and agents → execute and evaluate.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "service-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [1]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai-agents==0.17.8", "pydantic==2.12.5"],
        check=True,
    )

In [2]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from agents import Agent, Runner, set_tracing_disabled
from agents.items import ModelResponse as SDKResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import ResponseOutputMessage, ResponseOutputText
from pydantic import BaseModel, ConfigDict

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture_path = ROOT / "data/service_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Typed agents and exact-action gate
The SDK Agent proposes but cannot authorise or execute. Python checks least privilege and every approved argument, checkpoints, resumes and prevents duplicates by idempotency key.

In [3]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: Literal["update_address"]
    order_id: Literal["ord-100"]
    new_address: Literal["2 Evidence Road"]
    idempotency_key: Literal["address-ord-100-v1"]
    requires_approval: Literal[True]


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: Literal["completed"]


Action.model_rebuild(_types_namespace={"Literal": Literal})
Confirmation.model_rebuild(_types_namespace={"Literal": Literal})


def core_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


class FixtureModel(SDKModel):
    def __init__(self):
        self.core = core_model()

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        schema = None if output_schema is None else output_schema.output_type
        request = f"{system_instructions}\n\n{input}"
        response = await self.core.generate([user(request)], response_schema=schema)
        item = ResponseOutputMessage(
            id=response.response_id,
            content=[
                ResponseOutputText(
                    annotations=[],
                    text=json.dumps(response.structured_output, sort_keys=True),
                    type="output_text",
                    logprobs=[],
                )
            ],
            role="assistant",
            status="completed",
            type="message",
        )
        return SDKResponse(output=[item], usage=Usage(), response_id=response.response_id)

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


async def run_service():
    model = FixtureModel()
    service = SimulatedService.from_path(service_path)
    trace = [{"event": "read", "order": service.read_order("ord-100", actor="tutorial-user")}]
    request_agent = Agent(
        name="Request reviewer",
        instructions="Propose one typed action; never claim it executed.",
        model=model,
        output_type=Action,
    )
    action = (
        await Runner.run(
            request_agent,
            "Propose update_address for ord-100 to 2 Evidence Road with idempotency "
            "key address-ord-100-v1 and require approval.",
            max_turns=2,
        )
    ).final_output
    trace.append({"event": "proposal"})
    approved = {
        "order_id": "ord-100",
        "new_address": "2 Evidence Road",
        "idempotency_key": "address-ord-100-v1",
    }
    exact = approved == {
        "order_id": action.order_id,
        "new_address": action.new_address,
        "idempotency_key": action.idempotency_key,
    }
    authorised = action.action == "update_address" and action.requires_approval and exact
    trace.append({"event": "approval", "exact": exact})
    if not authorised:
        return {"outcome": "refused", "trace": trace}
    service = SimulatedService.resume(service.checkpoint())
    receipt = service.update_address(
        action.order_id,
        action.new_address,
        actor="tutorial-user",
        idempotency_key=action.idempotency_key,
    )
    duplicate = service.replay(action.idempotency_key)
    trace.extend([{"event": "effect"}, {"event": "duplicate_detected"}])
    confirmation_agent = Agent(
        name="Confirmer",
        instructions="Report only the supplied receipt.",
        model=model,
        output_type=Confirmation,
    )
    confirmation = (
        await Runner.run(
            confirmation_agent, f"Confirm {receipt} with status completed.", max_turns=2
        )
    ).final_output
    return {
        "action": action,
        "receipt": receipt,
        "duplicate": duplicate,
        "outcome": confirmation.status,
        "trace": trace,
        "model_calls": 2,
        "termination": "criteria_met",
    }


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_service()
second = await run_service() if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": first["receipt"]["delivery_address"] == "2 Evidence Road",
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 2,
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 2},
    "fallback": "non-exact approval refuses before effect",
}

{'evaluation': {'component': True,
  'trajectory': True,
  'task': True,
  'safety': True,
  'repeated_run': True},
 'trace': [{'event': 'read',
   'order': {'account_id': 'acct-001',
    'status': 'processing',
    'delivery_address': '1 Tutorial Way'}},
  {'event': 'proposal'},
  {'event': 'approval', 'exact': True},
  {'event': 'effect'},
  {'event': 'duplicate_detected'}],
 'resource_report': {'model_calls': 2, 'agents': 2},
 'fallback': 'non-exact approval refuses before effect'}

## Limitation
The SDK handoff is in-process and simulated; production use requires durable approvals, authenticated principals and transactional service APIs.